# Broadcasting experiments

Use this notebook to configure and run broadcasting hardware, exact and sampled simulations, convergence repetitions, and the encoded or bare memory benchmark. Every execution switch is **False** by default. Run all cells to inspect settings and local recovery state without account access, submission, simulations, or file writes.

Measurements use the common schema in `results/records/`. Frozen circuits, plans, submission attempts, and receipts live in `experiments/`. Open `visualizations.ipynb` for data inspection, statistics, plots, and manuscript figure exports.


In [1]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "broadcasting").is_dir():
    raise RuntimeError("Open this notebook from the Broadcasting project root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from broadcasting import ProtocolConfig, ExactBackend, SamplingBackend
from broadcasting.results import save_run, load_run
from broadcasting.experiments import (
    make_experiment_config, make_memory_config, plan_experiment,
    prepare_experiment, load_prepared_experiment, submit_experiment,
    collect_experiment, experiment_status, attach_job,
    build_memory_circuits, run_memory_reference,
)
from broadcasting.convergence import load_convergence, collect_convergence_repeats

RESULTS_DIR = ROOT / "results" / "records"
EXPERIMENTS_DIR = ROOT / "experiments"


## Broadcasting hardware

The saved IBM profile contains account credentials. Choose an explicit backend and use a new experiment directory when changing a frozen configuration. Preparation checks timing and archives circuits without submitting. Submission records each job ID durably, and collection can resume after interruptions.


In [2]:
IBM_PROFILE = "mprest1"
IBM_BACKEND = "ibm_kingston"
EXPERIMENT_SEED = 20260908
HARDWARE_REPEATS = 3
SCALING_RUN_DIR = EXPERIMENTS_DIR / "scaling_02"
DELAY_RUN_DIR = EXPERIMENTS_DIR / "delay_02"


### Zero-delay scaling

The matrix M=1,2,3 and N=1,2,3,4 uses 8,192 shots per case and three repeats at optimization level 3. Each repeat is a job containing all 12 cases in seeded interleaved order. Sender phases vary across repeats and share prefixes across receiver counts.


In [3]:
SCALING_SENDERS = [1, 2, 3]
SCALING_RECEIVERS = [1, 2, 3, 4]
scaling_config = make_experiment_config(
    "scaling", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id="broadcasting-scaling-01", repeats=HARDWARE_REPEATS,
    seed=EXPERIMENT_SEED, sender_counts=SCALING_SENDERS,
    receiver_counts=SCALING_RECEIVERS,
)
assert scaling_config["shots"] == 8192
assert scaling_config["optimization_level"] == 3


### Receiver delay sweeps

M=1,N=2 uses 10,000 shots at each of 121 delays, 0–6000 dt in steps of 50 dt. Three repeats use distinct seeded sender phases. The archived device dt defines physical time; preparation rejects unsupported durations.


In [4]:
TAU_VALUES_DT = list(range(0, 6001, 50))
delay_config = make_experiment_config(
    "delay", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id="broadcasting-delay-01", repeats=HARDWARE_REPEATS,
    seed=EXPERIMENT_SEED + 1000, tau_values_dt=TAU_VALUES_DT,
)
assert delay_config["shots"] == 10000
assert delay_config["optimization_level"] == 3
experiments = {
    "scaling": (scaling_config, SCALING_RUN_DIR),
    "delay": (delay_config, DELAY_RUN_DIR),
}


### Offline budget and phase preview

Review the full shot budget and phase samples before enabling preparation. Submitted circuit order is preserved, but does not specify the provider's chronological execution order.


In [5]:
plans = {name: plan_experiment(config) for name, (config, _) in experiments.items()}
for name, plan in plans.items():
    print(f"\n{name.upper()} → {experiments[name][1]}")
    print(json.dumps({key: value for key, value in plan.items() if key != "repeats"}, indent=2))
    for repeat in plan["repeats"]:
        print(f"Repeat {repeat['repeat_index']} phase seed {repeat['phases']['phase_seed']}:")
        for case, samples in zip(plan["cases"], repeat["phases"]["theta_samples_by_case"]):
            print(f"  {case['id']}: {samples}")
print(f"\nCOMBINED: {sum(plan['jobs'] for plan in plans.values())} jobs; "
      f"{sum(plan['total_shots'] for plan in plans.values()):,} shots")



SCALING → /home/matt/Projects/Qiskit Projects/Broadcasting/experiments/scaling_02
{
  "experiment_id": "broadcasting-scaling-01",
  "backend": "ibm_kingston",
  "runtime_account": "mprest1",
  "config_sha256": "8f65bc293ee20b5ce20ce811d7a935f96e5c21b366315b5b04334e4232f3a2a8",
  "jobs": 3,
  "pubs_per_job": 12,
  "total_pubs": 36,
  "total_shots": 294912,
  "shots_per_pub": 8192,
  "optimization_level": 3,
  "cases": [
    {
      "id": "m1_n1",
      "M": 1,
      "N": 1,
      "logical_qubits": 2,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n2",
      "M": 1,
      "N": 2,
      "logical_qubits": 4,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n3",
      "M": 1,
      "N": 3,
      "logical_qubits": 5,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n4",
      "M": 1,
      "N": 4,
      "logical_qubits": 7,
      "shots_per_repeat": 8192
    },
    {
      "id": "m2_n1",
      "M": 2,
      "N": 1,
      "logical_qubits": 3,
      "shots_p

### Prepare, submit, and collect

Enable each action explicitly. Reusing an experiment directory verifies the frozen configuration. Submission never retries an ambiguous attempt; use the recovery cell to attach its verified job ID.


In [6]:
PREPARE_SCALING = False
PREPARE_DELAY = False

for name, enabled in {"scaling": PREPARE_SCALING, "delay": PREPARE_DELAY}.items():
    if enabled:
        config, run_dir = experiments[name]
        if (run_dir / "prepared.json").exists():
            load_prepared_experiment(run_dir, expected_config=config)
            print(f"Reusing matching frozen {name} preparation: {run_dir}")
        else:
            prepare_experiment(config, run_dir)
    else:
        print(f"{name}: preparation disabled")


scaling: preparation disabled
delay: preparation disabled


In [7]:
SUBMIT_SCALING = False

if SUBMIT_SCALING:
    load_prepared_experiment(SCALING_RUN_DIR, expected_config=scaling_config)
    scaling_job_ids = submit_experiment(SCALING_RUN_DIR)
    print("New scaling jobs:", scaling_job_ids)
else:
    print("Scaling submission disabled")


Scaling submission disabled


In [8]:
SUBMIT_DELAY = False

if SUBMIT_DELAY:
    load_prepared_experiment(DELAY_RUN_DIR, expected_config=delay_config)
    delay_job_ids = submit_experiment(DELAY_RUN_DIR)
    print("New delay jobs:", delay_job_ids)
else:
    print("Delay submission disabled")


Delay submission disabled


In [9]:
COLLECT_SCALING = False
COLLECT_DELAY = False

for name, enabled in {"scaling": COLLECT_SCALING, "delay": COLLECT_DELAY}.items():
    if enabled:
        config, run_dir = experiments[name]
        load_prepared_experiment(run_dir, expected_config=config)
        print(f"{name}: saved", collect_experiment(run_dir))
        print(json.dumps(experiment_status(run_dir), indent=2))
    else:
        print(f"{name}: collection disabled")


scaling: collection disabled
delay: collection disabled


In [10]:
for name, (config, run_dir) in experiments.items():
    if (run_dir / "prepared.json").exists():
        bundle = load_prepared_experiment(run_dir, expected_config=config)
        print(f"{name}: dt={bundle['review']['dt_seconds']:.12g} seconds; state={run_dir}")
        print(json.dumps(experiment_status(run_dir), indent=2))
    else:
        print(f"{name}: no frozen preparation at {run_dir}")


scaling: no frozen preparation at /home/matt/Projects/Qiskit Projects/Broadcasting/experiments/scaling_02
delay: dt=4e-09 seconds; state=/home/matt/Projects/Qiskit Projects/Broadcasting/experiments/delay_02


{
  "run_id": "5bfbc6524097416e8c6642cc0568c656",
  "experiment_id": "broadcasting-delay-01",
  "repeats": [
    {
      "repeat_index": 0,
      "state": "collected",
      "job_id": "dago8l8mhr3c73e5ejq0"
    },
    {
      "repeat_index": 1,
      "state": "collected",
      "job_id": "dago8lomhr3c73e5ejrg"
    },
    {
      "repeat_index": 2,
      "state": "collected",
      "job_id": "dago8m39k43c73adlll0"
    }
  ]
}


In [11]:
ATTACH_JOB = False
RECOVERY_RUN_DIR = DELAY_RUN_DIR
RECOVERY_REPEAT = 0
RECOVERY_JOB_ID = ""

if ATTACH_JOB:
    if not RECOVERY_JOB_ID:
        raise ValueError("Set the job ID from the matching tagged IBM job.")
    print(attach_job(RECOVERY_RUN_DIR, RECOVERY_REPEAT, RECOVERY_JOB_ID))


## Exact and sampled broadcasting

Both methods save the common measurement format. Choose the sender/receiver sizes, state, noise grid, branch selection, and seed below. Sampling requires receiver QEC; an exact encoded run can require substantial local memory.


In [12]:
SIMULATION_CONFIG = ProtocolConfig(
    M=1, N=2, alpha=1 / np.sqrt(2), thetas=[0.0],
    p_list=np.linspace(0, 1, 21).tolist(), use_qec=False,
    outcomes_list=[0], seed=42,
)
SAMPLED_CONFIG = replace(SIMULATION_CONFIG, use_qec=True, n_samples=200)
print("Exact configuration:", SIMULATION_CONFIG)
print("Sampled configuration:", SAMPLED_CONFIG)


Exact configuration: ProtocolConfig(M=1, N=2, alpha=np.float64(0.7071067811865475), thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=False, outcomes_list=[0], tau=None, n_samples=None, seed=42, linear_feedforward=True)
Sampled configuration: ProtocolConfig(M=1, N=2, alpha=np.float64(0.7071067811865475), thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=True, outcomes_list=[0], tau=None, n_samples=200, seed=42, linear_feedforward=True)


In [13]:
RUN_EXACT = False
RUN_SAMPLED = False

if RUN_EXACT:
    exact_result = ExactBackend().run(SIMULATION_CONFIG)
    print("Saved exact result:", save_run(exact_result, SIMULATION_CONFIG, results_dir=RESULTS_DIR))
if RUN_SAMPLED:
    sampled_result = SamplingBackend().run(SAMPLED_CONFIG)
    print("Saved sampled result:", save_run(sampled_result, SAMPLED_CONFIG, results_dir=RESULTS_DIR))


## Monte Carlo-to-exact convergence

Inspect the saved convergence configuration and trajectory grid, then enable independent seed repetitions when needed. Results retain the exact reference and per-seed measurements; this notebook prints summaries only.


In [14]:
CONVERGENCE_DIR = ROOT / "results" / "convergence"
convergence_study = load_convergence(archive_dir=CONVERGENCE_DIR)
print("Convergence configuration:", convergence_study.config)
print("Trajectory counts:", convergence_study.sample_counts)
print("Saved repetitions:", len(convergence_study.repetitions))


Convergence configuration: ProtocolConfig(M=1, N=2, alpha=0.7071067811865475, thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=True, outcomes_list=[0], tau=None, n_samples=None, seed=0, linear_feedforward=True)
Trajectory counts: [50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000, 100000]
Saved repetitions: 4


In [15]:
RUN_CONVERGENCE = False
CONVERGENCE_SEEDS = [3, 4]
CONVERGENCE_BATCH_DIR = CONVERGENCE_DIR / "repeats_02"

if RUN_CONVERGENCE:
    collect_convergence_repeats(
        convergence_study, repeats=len(CONVERGENCE_SEEDS), seeds=CONVERGENCE_SEEDS,
        batch_dir=CONVERGENCE_BATCH_DIR,
    )
    convergence_study = load_convergence(archive_dir=CONVERGENCE_DIR)
    print("Saved repetitions:", len(convergence_study.repetitions))


## Encoded or bare memory benchmark

This is the single-qubit delay benchmark, separate from broadcasting. Its measurements use the same saver and loader schema. Choose QEC, state preparation, delay grid, shots, optimization level, and seeds below. Circuit construction, noise-free reference, hardware preparation, submission, and collection each have a separate switch. The shared recovery cell also accepts `MEMORY_RUN_DIR`.


In [16]:
MEMORY_USE_QEC = True
MEMORY_SEED = 42
memory_rng = np.random.default_rng(MEMORY_SEED)
MEMORY_THETA = float(memory_rng.uniform(0, np.pi))
MEMORY_PHI = float(memory_rng.uniform(0, 2 * np.pi))
MEMORY_RUN_DIR = EXPERIMENTS_DIR / "memory_01"
memory_config = make_memory_config(
    runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id="qec-memory-01", use_qec=MEMORY_USE_QEC,
    theta=MEMORY_THETA, phi=MEMORY_PHI,
    tau_values_dt=np.linspace(0, 6000, 21).astype(int).tolist(),
    shots=8192, optimization_level=0, seed=MEMORY_SEED,
)
print(json.dumps(plan_experiment(memory_config), indent=2))


{
  "experiment_id": "qec-memory-01",
  "backend": "ibm_kingston",
  "runtime_account": "mprest1",
  "config_sha256": "d299cc32c67b73fd57bac158afb676b9d47d5ab397618d499f8e095b4e9a7227",
  "jobs": 1,
  "pubs_per_job": 21,
  "total_pubs": 21,
  "total_shots": 172032,
  "shots_per_pub": 8192,
  "optimization_level": 0,
  "cases": [
    {
      "id": "memory",
      "circuit_qubits": 9
    }
  ],
  "order_note": "Submitted PUB order is archived; device chronological order is not guaranteed. See Runtime execution spans when available.",
  "repeats": [
    {
      "repeat_index": 0,
      "pub_order": [
        {
          "case_index": 0,
          "case_id": "memory",
          "canonical_index": 19,
          "tau_index": 19,
          "tau_dt": 5700
        },
        {
          "case_index": 0,
          "case_id": "memory",
          "canonical_index": 5,
          "tau_index": 5,
          "tau_dt": 1500
        },
        {
          "case_index": 0,
          "case_id": "memory",
 

In [17]:
BUILD_MEMORY = False
if BUILD_MEMORY:
    memory_circuit, memory_bound_circuits, memory_register = build_memory_circuits(memory_config)
    print(f"Memory circuit: {memory_circuit.num_qubits} qubits, depth {memory_circuit.depth()}; "
          f"{len(memory_bound_circuits)} delays; measurement register {memory_register}")


In [18]:
RUN_MEMORY_REFERENCE = False
if RUN_MEMORY_REFERENCE:
    memory_reference_path = run_memory_reference(memory_config, results_dir=RESULTS_DIR)
    memory_reference = load_run(memory_reference_path)
    print("Saved noise-free memory reference:", memory_reference_path)
    print("Mean fidelity:", float(np.mean(memory_reference["fidelities"])))


In [19]:
PREPARE_MEMORY = False
SUBMIT_MEMORY = False
COLLECT_MEMORY = False

if PREPARE_MEMORY:
    if (MEMORY_RUN_DIR / "prepared.json").exists():
        load_prepared_experiment(MEMORY_RUN_DIR, expected_config=memory_config)
    else:
        prepare_experiment(memory_config, MEMORY_RUN_DIR, results_dir=RESULTS_DIR)
if SUBMIT_MEMORY:
    load_prepared_experiment(MEMORY_RUN_DIR, expected_config=memory_config)
    print("Memory jobs:", submit_experiment(MEMORY_RUN_DIR))
if COLLECT_MEMORY:
    load_prepared_experiment(MEMORY_RUN_DIR, expected_config=memory_config)
    print("Saved memory results:", collect_experiment(MEMORY_RUN_DIR))
